# 04 - ViT Attention Analysis

This notebook studies where the ViT model looks before and after simple de-identification.

The implementation lives in `src/attention/reporting.py`, so this notebook stays focused on:

- loading cached predictions and metrics;
- selecting class-balanced visual examples;
- displaying attention maps, inverse-attention masking and landmark-guided summaries;
- keeping the analysis reproducible without redefining helper functions inside the notebook.

In [ ]:
from pathlib import Path
import importlib
import sys

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display
from PIL import Image

if Path.cwd().name == "notebooks":
    PROJECT_ROOT = Path.cwd().resolve().parent
else:
    PROJECT_ROOT = Path.cwd().resolve()

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

import src.configs as configs_module
import src.attention.reporting as attention_reporting

configs_module = importlib.reload(configs_module)
attention_reporting = importlib.reload(attention_reporting)

pd.set_option("display.max_columns", 140)
pd.set_option("display.width", 260)

print("Project root:", PROJECT_ROOT)
print("Dataset root:", configs_module.DEFAULT_DATA_ROOT)

## 4.1 Experiment Settings

The example set is intentionally class-balanced: one correctly classified original image per emotion, plus a small number of error cases when available.

This gives a more defensible qualitative analysis than repeatedly showing only the first few classes.

In [ ]:
EXAMPLE_SET_NAME = "class_balanced"

RECOMPUTE_PREDICTIONS = False
RESELECT_EXAMPLES = True
RECOMPUTE_ATTENTION_MAPS = False
RECOMPUTE_INVERSE_ATTENTION_MASKING = False
RECOMPUTE_LANDMARK_ATTENTION = False

ATTENTION_SCAN_MAX_IMAGES = None  # Use None for the full test set.
PREDICTION_BATCH_SIZE = 64
PREDICTION_NUM_WORKERS = 0

SOURCE_CONDITION_FOR_EXAMPLES = "Original"
CORRECT_EXAMPLES_PER_CLASS = 1
ERROR_EXAMPLES = 2

INVERSE_ATTENTION_MASK_FRACTION = 0.20
INVERSE_ATTENTION_MAX_EXAMPLES = 5

vit_conditions = attention_reporting.get_vit_conditions()
condition_labels = attention_reporting.condition_labels(vit_conditions)
output_paths = attention_reporting.AttentionOutputPaths.for_example_set(EXAMPLE_SET_NAME)
attention_reporting.ensure_attention_dirs(output_paths)

missing_checkpoints = attention_reporting.missing_checkpoint_labels(vit_conditions)

display(attention_reporting.fixed_filter_table())
print("Conditions:", condition_labels)
print("Output directory:", output_paths.attention_dir)
if missing_checkpoints:
    print("Missing ViT checkpoints:", missing_checkpoints)
else:
    print("All ViT checkpoints found.")

## 4.2 Context From Baseline And Privacy Experiments

This section only gives context. The full model comparison belongs to `02_Modeling.ipynb` and the fixed de-identification comparison belongs to `03_Privacy.ipynb`.

In [ ]:
baseline_df, deid_comparison_df, missing_baselines, missing_deid = attention_reporting.load_context_tables()

if missing_baselines:
    print("Missing baseline metrics:", missing_baselines)
if missing_deid:
    print("Missing fixed de-id metrics:", missing_deid)

if baseline_df.empty:
    print("No baseline metrics available.")
else:
    baseline_columns = [
        "model",
        "test_accuracy",
        "test_balanced_accuracy",
        "test_f1_macro",
        "test_precision_macro",
        "test_recall_macro",
        "best_epoch",
        "run_name",
    ]
    display(
        baseline_df[baseline_columns].style.format(
            {
                "test_accuracy": "{:.4f}",
                "test_balanced_accuracy": "{:.4f}",
                "test_f1_macro": "{:.4f}",
                "test_precision_macro": "{:.4f}",
                "test_recall_macro": "{:.4f}",
            }
        )
    )

vit_deid_df = attention_reporting.vit_deid_context(deid_comparison_df)
if vit_deid_df.empty:
    print("No ViT fixed de-id comparison available.")
else:
    display(
        vit_deid_df[
            [
                "model",
                "condition",
                "test_accuracy",
                "test_balanced_accuracy",
                "test_f1_macro",
                "delta_f1_macro",
                "test_precision_macro",
                "test_recall_macro",
                "run_name",
            ]
        ].style.format(
            {
                "test_accuracy": "{:.4f}",
                "test_balanced_accuracy": "{:.4f}",
                "test_f1_macro": "{:.4f}",
                "delta_f1_macro": "{:+.4f}",
                "test_precision_macro": "{:.4f}",
                "test_recall_macro": "{:.4f}",
            }
        )
    )

## 4.3 Test-Set Predictions For Attention Selection

Predictions are collected once for Original, Crop/context removal, Blur and Mosaic. The cached file is reused unless `RECOMPUTE_PREDICTIONS = True`.

In [ ]:
predictions_df, prediction_status = attention_reporting.load_or_collect_predictions(
    conditions=vit_conditions,
    paths=output_paths,
    recompute=RECOMPUTE_PREDICTIONS,
    data_root=configs_module.DEFAULT_DATA_ROOT,
    batch_size=PREDICTION_BATCH_SIZE,
    num_workers=PREDICTION_NUM_WORKERS,
    max_samples=ATTENTION_SCAN_MAX_IMAGES,
)

print(prediction_status)
prediction_summary_df = attention_reporting.prediction_summary(predictions_df)
if prediction_summary_df.empty:
    print("No predictions available.")
else:
    display(prediction_summary_df.style.format({"accuracy": "{:.4f}"}))

## 4.4 Class-Balanced Examples

The selected examples are used for qualitative maps, inverse-attention masking and landmark-guided summaries.

The goal is not to cherry-pick the best images, but to show how attention behaves across different expressions.

In [ ]:
selected_examples_df, selection_status = attention_reporting.load_or_select_examples(
    predictions=predictions_df,
    paths=output_paths,
    reselect=RESELECT_EXAMPLES,
    source_condition=SOURCE_CONDITION_FOR_EXAMPLES,
    correct_per_class=CORRECT_EXAMPLES_PER_CLASS,
    error_count=ERROR_EXAMPLES,
)

print(selection_status)
selected_summary_df = attention_reporting.selected_examples_summary(selected_examples_df)
if selected_summary_df.empty:
    print("No selected examples available.")
else:
    display(selected_summary_df)
    display(
        selected_examples_df[
            [
                "source_condition",
                "case_type",
                "true_class",
                "pred_class",
                "correct",
                "file_name",
                "image_path",
            ]
        ]
    )

## 4.5 Qualitative ViT Attention Maps

Each row compares the same base image under:

- Original;
- Crop/context removal;
- Blur;
- Mosaic.

For each condition, the notebook shows the transformed input and the ViT attention overlay.

In [ ]:
attention_metrics_df, attention_image_paths, attention_status = attention_reporting.load_or_create_attention_maps(
    selected_examples=selected_examples_df,
    conditions=vit_conditions,
    paths=output_paths,
    recompute=RECOMPUTE_ATTENTION_MAPS,
)

print(attention_status)
print("Attention metrics:", output_paths.attention_metrics_csv)

if attention_metrics_df.empty:
    print("No attention metrics available.")
else:
    display(attention_metrics_df.head())

for image_path in attention_image_paths:
    print(image_path.name)
    display(Image.open(image_path))

## 4.6 Inverse-Attention Masking Sanity Check

This checks whether high-attention regions are functionally important. It compares masking the most attended pixels with masking the least attended pixels.

This is not presented as a new privacy method; it is a lightweight explainability sanity check.

In [ ]:
inverse_attention_df, inverse_image_paths, inverse_status = (
    attention_reporting.load_or_create_inverse_attention_results(
        selected_examples=selected_examples_df,
        conditions=vit_conditions,
        paths=output_paths,
        recompute=RECOMPUTE_INVERSE_ATTENTION_MASKING,
        mask_fraction=INVERSE_ATTENTION_MASK_FRACTION,
        max_examples=INVERSE_ATTENTION_MAX_EXAMPLES,
    )
)

print(inverse_status)
if inverse_attention_df.empty:
    print("No inverse-attention results available.")
else:
    display(
        inverse_attention_df[
            [
                "example_index",
                "case_type",
                "true_class",
                "masked_region",
                "baseline_pred_class",
                "masked_pred_class",
                "prediction_changed",
                "baseline_pred_confidence",
                "masked_pred_confidence",
                "pred_confidence_drop",
                "true_class_confidence_drop",
            ]
        ].style.format(
            {
                "baseline_pred_confidence": "{:.3f}",
                "masked_pred_confidence": "{:.3f}",
                "pred_confidence_drop": "{:+.3f}",
                "true_class_confidence_drop": "{:+.3f}",
            }
        )
    )
    display(
        attention_reporting.inverse_attention_summary(inverse_attention_df).style.format(
            {
                "prediction_change_rate": "{:.3f}",
                "mean_pred_confidence_drop": "{:+.3f}",
                "mean_true_class_confidence_drop": "{:+.3f}",
            }
        )
    )

for image_path in inverse_image_paths:
    print(image_path.name)
    display(Image.open(image_path))

## 4.7 Landmark-Guided Attention Regions

This optional section uses MediaPipe Face Landmarker as anatomical reference.

Use it as supporting evidence only, because RAF-DB images are small and grayscale, and some transformations reduce landmark detectability.

In [ ]:
landmark_attention_df, landmark_plot_path, landmark_status = (
    attention_reporting.load_or_create_landmark_attention(
        selected_examples=selected_examples_df,
        conditions=vit_conditions,
        paths=output_paths,
        recompute=RECOMPUTE_LANDMARK_ATTENTION,
    )
)

print(landmark_status)
if landmark_attention_df.empty:
    print("No landmark attention results available.")
    print("Expected model path:", attention_reporting.LANDMARKER_MODEL_PATH)
else:
    detection_summary_df = attention_reporting.landmark_detection_summary(
        landmark_attention_df,
        condition_labels,
    )
    region_summary_df = attention_reporting.landmark_region_summary(
        landmark_attention_df,
        condition_labels,
    )
    display(
        detection_summary_df.style.format(
            {"detection_rate": "{:.3f}", "mean_landmarks": "{:.1f}"}
        )
    )
    if region_summary_df.empty:
        print("No detected faces available for region-level attention mass.")
    else:
        display(region_summary_df.style.format("{:.3f}"))

    if landmark_plot_path is not None and landmark_plot_path.exists():
        display(Image.open(landmark_plot_path))

## 4.8 Global Attention Metrics

Two simple metrics are used:

- normalized attention entropy: higher values mean attention is more diffuse;
- cosine similarity to the original attention map: higher values mean the transformed condition preserves the original attention pattern more closely.

In [ ]:
if attention_metrics_df.empty:
    print("No attention metrics available yet.")
else:
    attention_summary_df = attention_reporting.attention_metrics_summary(attention_metrics_df)
    display(
        attention_summary_df.style.format(
            {
                "entropy_mean": "{:.4f}",
                "entropy_std": "{:.4f}",
                "similarity_mean": "{:.4f}",
                "similarity_std": "{:.4f}",
            }
        )
    )

    fig = attention_reporting.plot_attention_metric_summary(
        attention_metrics=attention_metrics_df,
        labels=condition_labels,
        output_path=output_paths.attention_metrics_plot,
    )
    if fig is not None:
        plt.show()
        print("Saved attention metrics plot to:", output_paths.attention_metrics_plot)

## 4.9 Discussion Notes

Use this notebook to support the qualitative part of the report:

- Crop/context removal should preserve performance and attention structure better than the stronger perturbations.
- Blur may keep the class prediction but make attention more diffuse.
- Mosaic is expected to degrade both utility and interpretability.
- Landmark-guided results are useful as auxiliary evidence, but not as the main privacy metric.
- Inverse-attention masking helps test whether highlighted regions matter for prediction, but should be described as a sanity check because the sample is small.